# Master reproduction notebook

Reproduces every **cited** figure, table and number of the paper from the committed data in `results/data/`, in one top-to-bottom run:

| Section | Covers | Paper artifacts produced |
|---|---|---|
| 1. Setup | paths, plotting styles, check helper | none |
| 2. Data loading | real-channel, mock, synthetic and TMR data; **5-key snapshot for Table 5** | none (inputs) |
| 3. Fault models | Toeplitz / final-key model, reconciliation model, baseline term, leave-one-key-out, QBER dependence, leakage fit, zero-fault baseline | Table 4, χ² and CMH tests, b-values, 1.87x pool ratio, Pearson/Spearman, ξ₁/ξ₂ |
| 4. Mitigations | redundant PA, TMR, overheads | RPA 100% detection, TMR 0/100, 1.08x / 2.69x overhead |
| 5. Finite key | finite-key vs. asymptotic length | Table 5 |
| 6. Figures | regenerates the figures and pixel-compares them with the committed PNGs | Figs 4, 5, 6a, 6b, 7, 8, 9 |
| 7. Summary | every check in one table | none |

**How "reproduced" is decided.** Each number is recomputed here and compared with the value stored in the original notebook output / quoted in the paper (`PAPER_TRACEBACK.md`). Figures are regenerated into a temporary directory and compared pixel-by-pixel with `results/figures/`; the committed files are never overwritten.

**Known, intentional behaviours**
* Table 5 uses `results/data/key_pairs_metadata_table5_5keys.csv` (the original 5 keys). The live `key_pairs_metadata.csv` has since grown to 10 keys and would give 96.05% instead of 93.76%.
* Timing-based numbers (the 1.08x / 2.69x overheads) are re-measured, so they match only approximately, and are reported separately from the exact checks.
* Figures use an explicit matplotlib style per figure. The original `paper_figures_and_reconciliation_fit` notebook leaks `rcParams` between cells, which made Fig 6b depend on run order; here that dependency is removed.

## 1. Setup
*Covers:* imports, project paths, a temporary output directory for regenerated figures, and the helpers used to compare reproduced values with the expected ones.
*Produces:* no paper artifact.

In [1]:
import sys, json, time, tempfile, contextlib, io
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from scipy.optimize import curve_fit
from scipy.stats import chi2, norm, pearsonr, spearmanr, linregress, binom
from galois import GF2
from randextract import ToeplitzHashing

PROJECT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(PROJECT_DIR))

from qne.cascade import Key, ORIGINAL, Reconciliation, MockClassicalSession
from qne.cascade.fault_injection import SDCFaultInjector
from qne.cascade.key import key_from_sifted_json
from qne.cascade.finite_key import finite_key_output_length, h

DATA = PROJECT_DIR / "results" / "data"
FIGS = PROJECT_DIR / "results" / "figures"
RESULTS = PROJECT_DIR / "results"                # the embedded analysis code below addresses RESULTS / "data" / ... and RESULTS / "figures" / ...
REPRO = Path(tempfile.mkdtemp(prefix="master_repro_"))   # regenerated figures go here, never into results/
print("project:", PROJECT_DIR)
print("regenerated figures ->", REPRO)

# Style used by the original figure cells (paper_figures_and_reconciliation_fit c0).
STYLE_C0 = {
    "font.size": 16, "axes.labelsize": 18, "xtick.labelsize": 14, "ytick.labelsize": 14,
    "legend.fontsize": 14, "lines.linewidth": 2.5, "lines.markersize": 8,
}

CHECKS = []
def check(artifact, quantity, got, expected, tol=1e-6, kind="exact"):
    ok = bool(abs(float(got) - float(expected)) <= tol)
    CHECKS.append(dict(artifact=artifact, quantity=quantity, reproduced=float(got), expected=float(expected),
                       tol=tol, kind=kind, status="PASS" if ok else "MISMATCH"))
    print(f"[{'PASS' if ok else 'MISMATCH'}] {artifact}: {quantity} = {float(got):.6g} (expected {float(expected):.6g}, tol {tol:g})")

@contextlib.contextmanager
def quiet():
    with contextlib.redirect_stdout(io.StringIO()):
        yield

def compare_png(artifact, name):
    new, old = mpimg.imread(REPRO / name), mpimg.imread(FIGS / name)
    if new.shape != old.shape:
        diff = -1
    else:
        diff = int((np.abs(new - old).sum(axis=2) > 0).sum())
    CHECKS.append(dict(artifact=artifact, quantity=f"{name} differing pixels vs committed", reproduced=float(diff),
                       expected=0.0, tol=0.0, kind="figure", status="PASS" if diff == 0 else "MISMATCH"))
    print(f"[{'PASS' if diff == 0 else 'MISMATCH'}] {artifact}: {name} differing pixels vs committed = {diff}")

project: /Users/audreyfeng/research/qkd-dependability
regenerated figures -> /var/folders/bx/60xfygnd4gb520620446ccc80000gn/T/master_repro_d9ep4r2l


## 2. Data loading
*Covers:* every dataset the cited results are computed from. Real-channel Toeplitz / final-key sweeps (asymptotic length), the cleaned reconciliation sweep, the Mock sweep, the synthetic QBER sweep, the real-channel TMR runs, the zero-fault baseline, and the key metadata (live 10-key file **and** the original 5-key snapshot).
*Produces:* no paper artifact; asserts that the snapshot agrees with the live file on keys 0-4.

In [2]:
key_pairs_live = pd.read_csv(DATA / "key_pairs_metadata.csv")
key_pairs_table5 = pd.read_csv(DATA / "key_pairs_metadata_table5_5keys.csv")   # authoritative input for Table 5
num_cols = ["index", "m_sifted", "k_pe", "n_bits", "qber"]
assert len(key_pairs_table5) == 5 and len(key_pairs_live) == 10
assert (key_pairs_live[num_cols].iloc[:5].values == key_pairs_table5[num_cols].values).all(), "snapshot != live keys 0-4"
key_pairs_df = key_pairs_live            # name used by the embedded analysis code (keys 0-4 identical to the snapshot)

toeplitz = pd.read_csv(DATA / "sdc_realchannel_toeplitz_allkeys_asymptotic.csv")
final_key = pd.read_csv(DATA / "sdc_realchannel_finalkey_allkeys_asymptotic.csv")
reconciliation = pd.read_csv(DATA / "reconciliation_clean.csv")
mock = pd.read_csv(DATA / "mock_clean.csv")
mock_asym = mock[mock["length_mode"] == "asymptotic"]
synthetic_df = pd.read_csv(DATA / "synthetic_qber_sweep.csv")
tmr_real = pd.read_csv(DATA / "mitigation_tmr_realchannel.csv")
baseline = pd.read_csv(DATA / "check1_zerofault_validation.csv")

for name, df, n in [("toeplitz", toeplitz, 1200), ("final_key", final_key, 1200), ("reconciliation", reconciliation, 4000)]:
    check("Data", f"{name} rows", len(df), n)
check("Data", "TMR real-channel runs", len(tmr_real), 100)

[PASS] Data: toeplitz rows = 1200 (expected 1200, tol 1e-06)
[PASS] Data: final_key rows = 1200 (expected 1200, tol 1e-06)
[PASS] Data: reconciliation rows = 4000 (expected 4000, tol 1e-06)
[PASS] Data: TMR real-channel runs = 100 (expected 100, tol 1e-06)


## 3. Fault models
### 3a. Toeplitz-matrix and final-key faults: Table 4, χ² fits, CMH tests
*Covers:* silent-mismatch rate by firing probability for the two PA-stage faults, the χ² goodness-of-fit of the closed-form models `p·ℓ/(ℓ+t)·½` (Toeplitz) and `p·ℓ/(ℓ+t)` (final key), and the Cochran-Mantel-Haenszel real-vs-Mock tests.
*Produces:* **Table 4** (silent rates), the Toeplitz / final-key **χ²** results, the **CMH p-values**.

In [3]:
# Table 4: silent-mismatch rate at the three highest firing probabilities (sdc_sweeps_all5keys c193)
def silent_rate(df, prob):
    sub = df[df["prob"] == prob]
    return ((~sub["keys_match"].astype(bool)) & (sub["verification_passed"] == True)).mean()

table4 = pd.DataFrame({
    "prob": [0.1, 0.3, 0.5],
    "toeplitz": [silent_rate(toeplitz, p) for p in [0.1, 0.3, 0.5]],
    "final_key": [silent_rate(final_key, p) for p in [0.1, 0.3, 0.5]],
    "reconciliation": [silent_rate(reconciliation, p) for p in [0.1, 0.3, 0.5]],
})
print(table4.round(3).to_string(index=False))
for p, t, f in [(0.1, 0.033, 0.067), (0.3, 0.127, 0.233), (0.5, 0.213, 0.460)]:
    check("Table 4", f"toeplitz silent rate p={p}", silent_rate(toeplitz, p), t, tol=5e-4)
    check("Table 4", f"final_key silent rate p={p}", silent_rate(final_key, p), f, tol=5e-4)
    check("Table 4", f"reconciliation silent rate p={p}", silent_rate(reconciliation, p), 0.0, tol=5e-4)

 prob  toeplitz  final_key  reconciliation
  0.1     0.033      0.067             0.0
  0.3     0.127      0.233             0.0
  0.5     0.213      0.460             0.0
[PASS] Table 4: toeplitz silent rate p=0.1 = 0.0333333 (expected 0.033, tol 0.0005)
[PASS] Table 4: final_key silent rate p=0.1 = 0.0666667 (expected 0.067, tol 0.0005)
[PASS] Table 4: reconciliation silent rate p=0.1 = 0 (expected 0, tol 0.0005)
[PASS] Table 4: toeplitz silent rate p=0.3 = 0.126667 (expected 0.127, tol 0.0005)
[PASS] Table 4: final_key silent rate p=0.3 = 0.233333 (expected 0.233, tol 0.0005)
[PASS] Table 4: reconciliation silent rate p=0.3 = 0 (expected 0, tol 0.0005)
[PASS] Table 4: toeplitz silent rate p=0.5 = 0.213333 (expected 0.213, tol 0.0005)
[PASS] Table 4: final_key silent rate p=0.5 = 0.46 (expected 0.46, tol 0.0005)
[PASS] Table 4: reconciliation silent rate p=0.5 = 0 (expected 0, tol 0.0005)


In [4]:
def chi_square_fit_silent_only(df, predictor_fn):
    total_chi2, dof = 0.0, 0
    rows_out = []
    for prob in sorted(df["prob"].unique()):
        sub = df[df["prob"] == prob]
        n = len(sub)
        # observed = silent mismatches only (verification passed AND keys don't match)
        observed = ((~sub["keys_match"].astype(bool)) & (sub["verification_passed"] == True)).sum()
        predicted = sub.apply(lambda r: predictor_fn(prob, r["secure_key_length"], r["t_verify"]), axis=1).mean()
        p_clip = min(max(predicted, 1e-9), 1 - 1e-9)
        exp_m, exp_ok = n * p_clip, n * (1 - p_clip)
        total_chi2 += (observed - exp_m) ** 2 / exp_m + ((n - observed) - exp_ok) ** 2 / exp_ok
        dof += 1
        rows_out.append((prob, n, observed, n * p_clip, observed / n, p_clip))
    print(f"{'prob':<8}{'n':<6}{'observed':<10}{'predicted':<12}{'obs_rate':<10}{'pred_rate':<10}")
    for r in rows_out:
        print(f"{r[0]:<8}{r[1]:<6}{r[2]:<10}{r[3]:<12.2f}{r[4]:<10.4f}{r[5]:<10.4f}")
    return total_chi2, dof, 1 - chi2.cdf(total_chi2, dof)

def predicted_mismatch_toeplitz(prob, ell, t_verify):
    return prob * (ell / (ell + t_verify)) * 0.5
def predicted_mismatch_final_key(prob, ell, t_verify):
    return prob * (ell / (ell + t_verify))

print("=== TOEPLITZ, silent mismatches only ===")
stat, dof, p = chi_square_fit_silent_only(toeplitz, predicted_mismatch_toeplitz)
print(f"chi2={stat:.2f}, dof={dof}, p={p:.4f} -- {'REJECTED' if p<0.05 else 'not rejected'}")
print("=== FINAL KEY, silent mismatches only ===")
stat, dof, p = chi_square_fit_silent_only(final_key, predicted_mismatch_final_key)
print(f"chi2={stat:.2f}, dof={dof}, p={p:.4f} -- {'REJECTED' if p<0.05 else 'not rejected'}")

=== TOEPLITZ, silent mismatches only ===
prob    n     observed  predicted   obs_rate  pred_rate 
0.001   150   0         0.07        0.0000    0.0005    
0.003   150   0         0.22        0.0000    0.0015    
0.01    150   0         0.74        0.0000    0.0049    
0.03    150   0         2.22        0.0000    0.0148    
0.05    150   4         3.71        0.0267    0.0247    
0.1     150   5         7.42        0.0333    0.0494    
0.3     150   19        22.25       0.1267    0.1483    
0.5     150   32        37.08       0.2133    0.2472    
chi2=5.63, dof=8, p=0.6885 -- not rejected
=== FINAL KEY, silent mismatches only ===
prob    n     observed  predicted   obs_rate  pred_rate 
0.001   150   0         0.15        0.0000    0.0010    
0.003   150   0         0.44        0.0000    0.0030    
0.01    150   0         1.48        0.0000    0.0099    
0.03    150   1         4.45        0.0067    0.0297    
0.05    150   7         7.42        0.0467    0.0494    
0.1     150   10   

In [5]:
with quiet():
    t_stat, t_dof, t_p = chi_square_fit_silent_only(toeplitz, predicted_mismatch_toeplitz)
    f_stat, f_dof, f_p = chi_square_fit_silent_only(final_key, predicted_mismatch_final_key)
check("Toeplitz/final-key chi-square", "Toeplitz chi2", t_stat, 5.63, tol=5e-3)
check("Toeplitz/final-key chi-square", "Toeplitz dof", t_dof, 8)
check("Toeplitz/final-key chi-square", "Toeplitz p", t_p, 0.6885, tol=5e-5)
check("Toeplitz/final-key chi-square", "final-key chi2", f_stat, 10.21, tol=5e-3)
check("Toeplitz/final-key chi-square", "final-key p", f_p, 0.2509, tol=5e-5)

[PASS] Toeplitz/final-key chi-square: Toeplitz chi2 = 5.63074 (expected 5.63, tol 0.005)
[PASS] Toeplitz/final-key chi-square: Toeplitz dof = 8 (expected 8, tol 1e-06)
[PASS] Toeplitz/final-key chi-square: Toeplitz p = 0.688516 (expected 0.6885, tol 5e-05)
[PASS] Toeplitz/final-key chi-square: final-key chi2 = 10.2055 (expected 10.21, tol 0.005)
[PASS] Toeplitz/final-key chi-square: final-key p = 0.250901 (expected 0.2509, tol 5e-05)


In [6]:
import pandas as pd
import numpy as np
from scipy.stats import chi2

toeplitz = pd.read_csv(PROJECT_DIR / "results" / "data" / "sdc_realchannel_toeplitz_allkeys_asymptotic.csv")
final_key = pd.read_csv(PROJECT_DIR / "results" / "data" / "sdc_realchannel_finalkey_allkeys_asymptotic.csv")
mock = pd.read_csv(PROJECT_DIR / "results" / "data" / "mock_clean.csv")
mock_asym = mock[mock["length_mode"] == "asymptotic"]

def cmh_manual(tables):
    num, den = 0.0, 0.0
    for t in tables:
        a, b, c, d = t[0,0], t[0,1], t[1,0], t[1,1]
        n = a+b+c+d
        if n < 2: continue
        row1, row2, col1, col2 = a+b, c+d, a+c, b+d
        E_a = row1*col1/n
        var_a = row1*row2*col1*col2 / (n**2*(n-1))
        num += (a - E_a); den += var_a
    stat = num**2/den if den else np.nan
    return stat, (1 - chi2.cdf(stat, 1)) if den else np.nan

In [7]:
def build_tables_silent_both(real_df, mock_df, fault_type):
    tables = []
    for p in sorted(real_df["prob"].unique()):
        r = real_df[real_df["prob"]==p]
        m = mock_df[(mock_df["prob"]==p) & (mock_df["fault_type"]==fault_type)]

        r_mis = ((~r["keys_match"].astype(bool)) & (r["verification_passed"]==True)).sum()
        r_n = len(r)

        m_mis = ((~m["keys_match"].astype(bool)) & (m["verification_passed"]==True)).sum()
        m_n = len(m)

        tables.append(np.array([[r_mis, r_n-r_mis],[m_mis, m_n-m_mis]]))
    return tables

for name, df, ftype in [("toeplitz", toeplitz, "toeplitz"), ("final_key", final_key, "final_key")]:
    tables = build_tables_silent_both(df, mock_asym, ftype)
    stat, p = cmh_manual(tables)
    n_real = sum(t[0].sum() for t in tables)
    n_mock = sum(t[1].sum() for t in tables)
    print(f"{name} (silent-only both sides, n_real={n_real}, n_mock={n_mock}): "
          f"CMH stat={stat:.3f}, p={p:.4f} -- {'significant' if p<0.05 else 'not significant'}")

toeplitz (silent-only both sides, n_real=1200, n_mock=800): CMH stat=1.946, p=0.1630 -- not significant
final_key (silent-only both sides, n_real=1200, n_mock=800): CMH stat=0.978, p=0.3228 -- not significant


In [8]:
tb_t = build_tables_silent_both(toeplitz, mock_asym, "toeplitz")
tb_f = build_tables_silent_both(final_key, mock_asym, "final_key")
cmh_t, cmh_t_p = cmh_manual(tb_t)
cmh_f, cmh_f_p = cmh_manual(tb_f)
check("CMH (real vs Mock)", "Toeplitz statistic", cmh_t, 1.946, tol=5e-4)
check("CMH (real vs Mock)", "Toeplitz p", cmh_t_p, 0.1630, tol=5e-5)
check("CMH (real vs Mock)", "final-key statistic", cmh_f, 0.978, tol=5e-4)
check("CMH (real vs Mock)", "final-key p", cmh_f_p, 0.3228, tol=5e-5)

[PASS] CMH (real vs Mock): Toeplitz statistic = 1.94573 (expected 1.946, tol 0.0005)
[PASS] CMH (real vs Mock): Toeplitz p = 0.163048 (expected 0.163, tol 5e-05)
[PASS] CMH (real vs Mock): final-key statistic = 0.97766 (expected 0.978, tol 0.0005)
[PASS] CMH (real vs Mock): final-key p = 0.322777 (expected 0.3228, tol 5e-05)


### 3b. Reconciliation-state faults: pool-size model, 1.87x ratio, baseline term, leave-one-key-out
*Covers:* `P(mismatch) = 1 - (1-p)^K` with `K = total_corrections` (rejected overall), the naive `n·QBER` pool estimate vs. the measured one, the per-key baseline-error term `b` and its χ², and leave-one-key-out generalisation of `c ~ QBER`.
*Produces:* the **χ² = 93.05** rejection, the **1.87x pool-size ratio**, the **b-values and χ² = 7.91**, and the **leave-one-key-out MAE 0.036 / 0.011 and χ² = 7.65**.

In [9]:
def predicted_mismatch_totalcorrections(prob, total_corrections):
    return 1 - (1-prob)**total_corrections

def chi_square_fit_recon_totalcorrections(df):
    total_chi2, dof = 0.0, 0
    rows_out = []
    for prob in sorted(df["prob"].unique()):
        sub = df[df["prob"] == prob]
        n_trials = len(sub)
        observed = (~sub["keys_match"].astype(bool)).sum()

        pool_size = sub["total_corrections"].mean()
        predicted = 1 - (1-prob)**pool_size

        p_clip = min(max(predicted, 1e-9), 1 - 1e-9)
        exp_m, exp_ok = n_trials*p_clip, n_trials*(1-p_clip)
        total_chi2 += (observed-exp_m)**2/exp_m + ((n_trials-observed)-exp_ok)**2/exp_ok
        dof += 1
        rows_out.append((prob, n_trials, observed, n_trials*p_clip, observed/n_trials, p_clip, pool_size))

    print(f"{'prob':<8}{'n':<6}{'observed':<10}{'predicted':<12}{'obs_rate':<10}{'pred_rate':<10}{'pool_size':<10}")
    for r in rows_out:
        print(f"{r[0]:<8}{r[1]:<6}{r[2]:<10}{r[3]:<12.2f}{r[4]:<10.4f}{r[5]:<10.4f}{r[6]:<10.2f}")
    return total_chi2, dof, 1-chi2.cdf(total_chi2, dof)

print("=== Reconciliation, pool_size = actual total_corrections (not derived) ===")
stat, dof, p = chi_square_fit_recon_totalcorrections(reconciliation)
print(f"chi2={stat:.2f}, dof={dof}, p={p:.4f} -- {'REJECTED' if p<0.05 else 'not rejected'}")

=== Reconciliation, pool_size = actual total_corrections (not derived) ===
prob    n     observed  predicted   obs_rate  pred_rate pool_size 
0.001   500   45        17.01       0.0900    0.0340    34.60     
0.003   500   90        49.39       0.1800    0.0988    34.62     
0.01    500   174       146.84      0.3480    0.2937    34.59     
0.03    500   326       325.81      0.6520    0.6516    34.62     
0.05    500   422       415.67      0.8440    0.8313    34.70     
0.1     500   483       485.96      0.9660    0.9719    33.91     
0.3     500   500       499.99      1.0000    1.0000    30.60     
0.5     500   500       500.00      1.0000    1.0000    27.25     
chi2=93.05, dof=8, p=0.0000 -- REJECTED


In [10]:
with quiet():
    rc_stat, rc_dof, rc_p = chi_square_fit_recon_totalcorrections(reconciliation)
check("Reconciliation model", "chi2 (pool = total_corrections)", rc_stat, 93.05, tol=5e-3)
check("Reconciliation model", "dof", rc_dof, 8)
check("Reconciliation model", "p (rejected)", rc_p, 0.0, tol=1e-4)

[PASS] Reconciliation model: chi2 (pool = total_corrections) = 93.0504 (expected 93.05, tol 0.005)
[PASS] Reconciliation model: dof = 8 (expected 8, tol 1e-06)
[PASS] Reconciliation model: p (rejected) = 1.11022e-16 (expected 0, tol 0.0001)


In [11]:
n_sifted_estimate = reconciliation["k_pe"] * 10
implied_pool = (n_sifted_estimate * reconciliation["qber"])
print("total_corrections vs. naive n*QBER pool estimate:")
print(pd.DataFrame({
    "total_corrections": reconciliation["total_corrections"],
    "naive_pool_estimate": implied_pool,
    "ratio": reconciliation["total_corrections"] / implied_pool
}).describe())

total_corrections vs. naive n*QBER pool estimate:
       total_corrections  naive_pool_estimate        ratio
count        4000.000000          4000.000000  4000.000000
mean           33.110000            30.000000     1.870877
std             4.739622            20.978799     1.250702
min             8.000000            10.000000     0.500000
25%            33.000000            10.000000     0.660000
50%            35.000000            20.000000     1.750000
75%            35.000000            50.000000     3.500000
max            37.000000            60.000000     3.700000


In [12]:
ratio = reconciliation["total_corrections"] / (reconciliation["k_pe"] * 10 * reconciliation["qber"])
check("Pool-size ratio", "mean total_corrections / (n*QBER)", ratio.mean(), 1.87, tol=5e-3)
check("Pool-size ratio", "rows pooled", len(ratio), 4000)
per_key_ratio = ratio.groupby(reconciliation["key_index"]).mean()
print("per-key ratio (pooled mean hides keys 0 and 1 falling below 1x):")
print(per_key_ratio.round(3).to_string())

[PASS] Pool-size ratio: mean total_corrections / (n*QBER) = 1.87088 (expected 1.87, tol 0.005)
[PASS] Pool-size ratio: rows pooled = 4000 (expected 4000, tol 1e-06)
per-key ratio (pooled mean hides keys 0 and 1 falling below 1x):
key_index
0    0.656
1    0.581
2    3.250
3    3.196
4    1.671


In [13]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.optimize import curve_fit

# --- Setup: paths and data loading ---
PROJECT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RESULTS = PROJECT_DIR / "results"

key_pairs_df = pd.read_csv(str(RESULTS / "data" / "key_pairs_metadata.csv"))

reconciliation_clean = pd.read_csv(str(RESULTS / "data" / "reconciliation_clean.csv"))
reconciliation_clean["reconciliation_prob"] = reconciliation_clean["reconciliation_prob"].astype(float)
reconciliation_clean["key_index"] = reconciliation_clean["key_index"].astype(float).astype(int)
reconciliation_clean["mismatch"] = reconciliation_clean["keys_match"].astype(str).isin(["False", "0", "0.0"]).astype(float)

print(f"loaded {len(reconciliation_clean)} rows across {reconciliation_clean['key_index'].nunique()} keys")
print(reconciliation_clean.groupby(["key_index", "reconciliation_prob"]).size().describe())

loaded 4000 rows across 5 keys
count     40.0
mean     100.0
std        0.0
min      100.0
25%      100.0
50%      100.0
75%      100.0
max      100.0
dtype: float64


In [14]:
# --- The model and fit ---
def model(p, K):
    """P(mismatch) = 1 - (1-p)^K -- K is the free-fit 'effective pool size'."""
    return 1 - (1 - p) ** K

results = []
for key_idx in sorted(reconciliation_clean["key_index"].unique()):
    sub = reconciliation_clean[reconciliation_clean["key_index"] == key_idx]
    probs = sub.groupby("reconciliation_prob")["mismatch"].mean()
    p_vals = probs.index.values
    mismatch_vals = probs.values

    krow = key_pairs_df.iloc[key_idx]
    K_theory = krow["n_bits"] * krow["qber"]

    try:
        K_fit, K_cov = curve_fit(model, p_vals, mismatch_vals, p0=[max(K_theory, 5)], bounds=(0, np.inf))
        K_fit = K_fit[0]
        K_err = np.sqrt(K_cov[0, 0])
    except RuntimeError as e:
        print(f"key{key_idx}: fit failed ({e}) -- try a different p0")
        K_fit, K_err = np.nan, np.nan

    results.append({
        "key_index": key_idx,
        "K_fit": K_fit,
        "K_fit_stderr": K_err,
        "K_theory": K_theory,
        "ratio_fit_over_theory": K_fit / K_theory if K_theory > 0 else np.nan,
    })

df_fit = pd.DataFrame(results)
print(df_fit.to_string(index=False))

 key_index     K_fit  K_fit_stderr  K_theory  ratio_fit_over_theory
         0 33.241715      1.125465 44.923077               0.739970
         1 32.911683      0.711796 53.968338               0.609833
         2 51.261022      8.218832  8.983957               5.705840
         3 42.522735      4.834779  8.981675               4.734388
         4 35.836049      1.528762 17.953247               1.996076


In [15]:
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

def model_with_baseline(p, b, c):
    return 1 - (1 - b) * (1 - p) ** c

results = []
for key_idx in sorted(reconciliation_clean["key_index"].unique()):
    sub = reconciliation_clean[reconciliation_clean["key_index"] == key_idx]

    # c is FIXED to this key's actual measured total_corrections -- not fit, not theoretical
    c_fixed = sub["total_corrections"].mean()

    probs = sub.groupby("reconciliation_prob")["mismatch"].mean()
    p_vals = probs.index.values
    mismatch_vals = probs.values

    def model_fixed_c(p, b):
        return model_with_baseline(p, b, c_fixed)

    try:
        b_fit, b_cov = curve_fit(model_fixed_c, p_vals, mismatch_vals, p0=[0.05], bounds=(0, 1))
        b_fit = b_fit[0]
        b_err = np.sqrt(b_cov[0, 0])
    except RuntimeError as e:
        print(f"key{key_idx}: fit failed ({e})")
        b_fit, b_err = np.nan, np.nan

    results.append({
        "key_index": key_idx,
        "c_fixed (measured total_corrections)": c_fixed,
        "b_fit (baseline error rate)": b_fit,
        "b_fit_stderr": b_err,
    })

df_baseline_fit = pd.DataFrame(results)
print(df_baseline_fit.to_string(index=False))

 key_index  c_fixed (measured total_corrections)  b_fit (baseline error rate)  b_fit_stderr
         0                              32.79625                 1.992909e-02      0.009553
         1                              34.86875                 3.906122e-07      0.011002
         2                              32.50125                 1.925303e-01      0.013850
         3                              31.96000                 1.353539e-01      0.009955
         4                              33.42375                 4.359409e-02      0.006075


In [16]:
# b-values (baseline error rate) per key, from the fit above
b_expected = {0: 1.99e-2, 1: 3.9e-7, 2: 1.925e-1, 3: 1.354e-1, 4: 4.36e-2}
for k, v in b_expected.items():
    got = float(df_baseline_fit.loc[df_baseline_fit["key_index"] == k, "b_fit (baseline error rate)"].iloc[0])
    check("b-values", f"b (key {k})", got, v, tol=max(5e-4, 0.0))

[PASS] b-values: b (key 0) = 0.0199291 (expected 0.0199, tol 0.0005)
[PASS] b-values: b (key 1) = 3.90612e-07 (expected 3.9e-07, tol 0.0005)
[PASS] b-values: b (key 2) = 0.19253 (expected 0.1925, tol 0.0005)
[PASS] b-values: b (key 3) = 0.135354 (expected 0.1354, tol 0.0005)
[PASS] b-values: b (key 4) = 0.0435941 (expected 0.0436, tol 0.0005)


In [17]:
import numpy as np
import pandas as pd
from scipy.stats import chi2

def chi_square_reconciliation_baseline_model(reconciliation_clean, key_pairs_df, df_baseline_fit):
    chi2_stat, dof = 0.0, 0
    rows = []

    for key_idx in sorted(reconciliation_clean["key_index"].unique()):
        sub = reconciliation_clean[reconciliation_clean["key_index"] == key_idx]
        c_fixed = sub["total_corrections"].mean()
        b_fit = df_baseline_fit.loc[df_baseline_fit["key_index"] == key_idx, "b_fit (baseline error rate)"].values[0]

        key_chi2, key_dof = 0.0, 0
        for prob in sorted(sub["reconciliation_prob"].unique()):
            trials = sub[sub["reconciliation_prob"] == prob]
            n_trials = len(trials)
            observed_mismatch = trials["mismatch"].sum()

            predicted_p = 1 - (1 - b_fit) * (1 - prob) ** c_fixed
            predicted_p = min(max(predicted_p, 1e-6), 1 - 1e-6)

            expected_mismatch = n_trials * predicted_p
            expected_match = n_trials * (1 - predicted_p)
            observed_match = n_trials - observed_mismatch

            key_chi2 += (observed_mismatch - expected_mismatch) ** 2 / expected_mismatch
            key_chi2 += (observed_match - expected_match) ** 2 / expected_match
            key_dof += 1

        p_value = 1 - chi2.cdf(key_chi2, key_dof)
        rows.append({"key_index": key_idx, "c": c_fixed, "b": b_fit,
                      "chi2": key_chi2, "dof": key_dof, "p_value": p_value})
        chi2_stat += key_chi2
        dof += key_dof

    per_key_df = pd.DataFrame(rows)
    overall_p = 1 - chi2.cdf(chi2_stat, dof)

    print(per_key_df.to_string(index=False))
    print(f"\nOverall (pooled across keys): chi2={chi2_stat:.2f}, dof={dof}, p={overall_p:.4f}")
    print("NOT rejected" if overall_p > 0.05 else "REJECTED")
    return per_key_df, chi2_stat, dof, overall_p

per_key_chi2, chi2_stat, dof, overall_p = chi_square_reconciliation_baseline_model(
    reconciliation_clean, key_pairs_df, df_baseline_fit
)

 key_index        c            b     chi2  dof  p_value
         0 32.79625 1.992909e-02 1.356261    8 0.994845
         1 34.86875 3.906122e-07 2.042651    8 0.979676
         2 32.50125 1.925303e-01 2.064671    8 0.978965
         3 31.96000 1.353539e-01 1.410928    8 0.994090
         4 33.42375 4.359409e-02 1.039658    8 0.997985

Overall (pooled across keys): chi2=7.91, dof=40, p=1.0000
NOT rejected


In [18]:
check("b-values / chi-square", "pooled chi2 (baseline model)", chi2_stat, 7.91, tol=5e-3)
check("b-values / chi-square", "dof", dof, 40)
check("b-values / chi-square", "p", overall_p, 1.0, tol=5e-5)

[PASS] b-values / chi-square: pooled chi2 (baseline model) = 7.91417 (expected 7.91, tol 0.005)
[PASS] b-values / chi-square: dof = 40 (expected 40, tol 1e-06)
[PASS] b-values / chi-square: p = 1 (expected 1, tol 5e-05)


In [19]:
import numpy as np
import pandas as pd
from scipy.stats import chi2, linregress

def leave_one_key_out_c_generalization(reconciliation_clean, key_pairs_df):
    """Leave-one-key-out: fit c ~ a + slope*QBER on 4 keys, predict c (and
    therefore P(D)) for the held-out key's QBER, compare against its
    actually observed mismatch rates. No baseline term b here -- this test
    is specifically about whether c's relationship to QBER generalizes,
    independent of the separate baseline-error question."""

    # Per-key (QBER, measured c) points -- the regression dataset
    per_key = []
    for key_idx in sorted(reconciliation_clean["key_index"].unique()):
        sub = reconciliation_clean[reconciliation_clean["key_index"] == key_idx]
        c_measured = sub["total_corrections"].mean()
        qber = key_pairs_df.iloc[key_idx]["qber"]
        per_key.append({"key_index": key_idx, "qber": qber, "c_measured": c_measured})
    per_key_df = pd.DataFrame(per_key)

    all_rows = []
    fold_summaries = []

    for held_out_idx in per_key_df["key_index"]:
        train = per_key_df[per_key_df["key_index"] != held_out_idx]
        test_row = per_key_df[per_key_df["key_index"] == held_out_idx].iloc[0]

        # Fit c ~ a + slope*QBER on the other 4 keys
        fit = linregress(train["qber"], train["c_measured"])
        c_pred = fit.intercept + fit.slope * test_row["qber"]

        fold_summaries.append({
            "held_out_key": held_out_idx, "intercept": fit.intercept, "slope": fit.slope,
            "c_measured": test_row["c_measured"], "c_pred": c_pred,
        })

        # Predict mismatch rate at each firing probability for the held-out key,
        # using c_pred instead of the directly measured c
        sub = reconciliation_clean[reconciliation_clean["key_index"] == held_out_idx]
        for prob in sorted(sub["reconciliation_prob"].unique()):
            trials = sub[sub["reconciliation_prob"] == prob]
            n_trials = len(trials)
            observed_rate = trials["mismatch"].mean()
            predicted_rate = 1 - (1 - prob) ** c_pred

            all_rows.append({
                "held_out_key": held_out_idx, "prob": prob, "n_trials": n_trials,
                "observed_rate": observed_rate, "predicted_rate": predicted_rate,
                "abs_error": abs(observed_rate - predicted_rate),
            })

    results_df = pd.DataFrame(all_rows)
    fold_df = pd.DataFrame(fold_summaries)

    mae = results_df["abs_error"].mean()

    # Chi-square across all held-out predictions, pooled
    chi2_stat, dof = 0.0, 0
    for _, row in results_df.iterrows():
        n = row["n_trials"]
        p_pred = min(max(row["predicted_rate"], 1e-6), 1 - 1e-6)
        observed_mismatch = row["observed_rate"] * n
        expected_mismatch = n * p_pred
        expected_match = n * (1 - p_pred)
        observed_match = n - observed_mismatch
        chi2_stat += (observed_mismatch - expected_mismatch) ** 2 / expected_mismatch
        chi2_stat += (observed_match - expected_match) ** 2 / expected_match
        dof += 1

    p_value = 1 - chi2.cdf(chi2_stat, dof)

    print("Per-fold regression (train on 4 keys, predict held-out):")
    print(fold_df.to_string(index=False))
    print(f"\nMean absolute error (predicted vs. observed mismatch rate): {mae:.4f}")
    print(f"Chi-square (pooled, held-out predictions): chi2={chi2_stat:.2f}, dof={dof}, p={p_value:.4f}")
    print("NOT rejected -- generalizes" if p_value > 0.05 else "REJECTED")

    return results_df, fold_df, mae, chi2_stat, dof, p_value

results_df, fold_df, mae, chi2_stat, dof, p_value = leave_one_key_out_c_generalization(
    reconciliation_clean, key_pairs_df
)

Per-fold regression (train on 4 keys, predict held-out):
 held_out_key  intercept      slope  c_measured    c_pred
            0  31.923741 192.221081    32.79625 34.388114
            1  32.400798  46.254836    34.86875 33.133065
            2  31.939588 145.084527    32.50125 32.327514
            3  32.346086 115.159515    31.96000 32.647551
            4  31.716467 154.975863    33.42375 32.521536

Mean absolute error (predicted vs. observed mismatch rate): 0.0359
Chi-square (pooled, held-out predictions): chi2=209.15, dof=40, p=0.0000
REJECTED


In [20]:
check("Leave-one-key-out (no baseline)", "MAE", mae, 0.0359, tol=5e-5)
check("Leave-one-key-out (no baseline)", "chi2", chi2_stat, 209.15, tol=5e-3)

[PASS] Leave-one-key-out (no baseline): MAE = 0.0358796 (expected 0.0359, tol 5e-05)
[PASS] Leave-one-key-out (no baseline): chi2 = 209.154 (expected 209.15, tol 0.005)


In [21]:
def leave_one_key_out_c_generalization_with_baseline(reconciliation_clean, key_pairs_df, df_baseline_fit):
    per_key = []
    for key_idx in sorted(reconciliation_clean["key_index"].unique()):
        sub = reconciliation_clean[reconciliation_clean["key_index"] == key_idx]
        c_measured = sub["total_corrections"].mean()
        qber = key_pairs_df.iloc[key_idx]["qber"]
        b = df_baseline_fit.loc[df_baseline_fit["key_index"] == key_idx, "b_fit (baseline error rate)"].values[0]
        per_key.append({"key_index": key_idx, "qber": qber, "c_measured": c_measured, "b": b})
    per_key_df = pd.DataFrame(per_key)

    all_rows, fold_summaries = [], []
    for held_out_idx in per_key_df["key_index"]:
        train = per_key_df[per_key_df["key_index"] != held_out_idx]
        test_row = per_key_df[per_key_df["key_index"] == held_out_idx].iloc[0]

        fit = linregress(train["qber"], train["c_measured"])
        c_pred = fit.intercept + fit.slope * test_row["qber"]
        b_held_out = test_row["b"]  # use this key's own already-fit b, unchanged

        fold_summaries.append({"held_out_key": held_out_idx, "c_measured": test_row["c_measured"],
                                  "c_pred": c_pred, "b": b_held_out})

        sub = reconciliation_clean[reconciliation_clean["key_index"] == held_out_idx]
        for prob in sorted(sub["reconciliation_prob"].unique()):
            trials = sub[sub["reconciliation_prob"] == prob]
            n_trials = len(trials)
            observed_rate = trials["mismatch"].mean()
            predicted_rate = 1 - (1 - b_held_out) * (1 - prob) ** c_pred
            all_rows.append({"held_out_key": held_out_idx, "prob": prob, "n_trials": n_trials,
                              "observed_rate": observed_rate, "predicted_rate": predicted_rate,
                              "abs_error": abs(observed_rate - predicted_rate)})

    results_df = pd.DataFrame(all_rows)
    mae = results_df["abs_error"].mean()

    chi2_stat, dof = 0.0, 0
    for _, row in results_df.iterrows():
        n = row["n_trials"]
        p_pred = min(max(row["predicted_rate"], 1e-6), 1 - 1e-6)
        observed_mismatch = row["observed_rate"] * n
        expected_mismatch, expected_match = n * p_pred, n * (1 - p_pred)
        observed_match = n - observed_mismatch
        chi2_stat += (observed_mismatch - expected_mismatch) ** 2 / expected_mismatch
        chi2_stat += (observed_match - expected_match) ** 2 / expected_match
        dof += 1
    p_value = 1 - chi2.cdf(chi2_stat, dof)

    print(pd.DataFrame(fold_summaries).to_string(index=False))
    print(f"\nMAE: {mae:.4f}")
    print(f"chi2={chi2_stat:.2f}, dof={dof}, p={p_value:.4f}")
    print("NOT rejected" if p_value > 0.05 else "REJECTED")
    return results_df, mae, chi2_stat, dof, p_value

loo_results, loo_mae, loo_chi2, loo_dof, loo_p = leave_one_key_out_c_generalization_with_baseline(reconciliation_clean, key_pairs_df, df_baseline_fit)

 held_out_key  c_measured    c_pred            b
            0    32.79625 34.388114 1.992909e-02
            1    34.86875 33.133065 3.906122e-07
            2    32.50125 32.327514 1.925303e-01
            3    31.96000 32.647551 1.353539e-01
            4    33.42375 32.521536 4.359409e-02

MAE: 0.0107
chi2=7.65, dof=40, p=1.0000
NOT rejected


In [22]:
check("Leave-one-key-out (with baseline)", "MAE", loo_mae, 0.0107, tol=5e-5)
check("Leave-one-key-out (with baseline)", "chi2", loo_chi2, 7.65, tol=5e-3)
check("Leave-one-key-out (with baseline)", "dof", loo_dof, 40)

[PASS] Leave-one-key-out (with baseline): MAE = 0.0106578 (expected 0.0107, tol 5e-05)
[PASS] Leave-one-key-out (with baseline): chi2 = 7.64607 (expected 7.65, tol 0.005)
[PASS] Leave-one-key-out (with baseline): dof = 40 (expected 40, tol 1e-06)


### 3c. QBER dependence (synthetic sweep)
*Covers:* correlation between true QBER and the reconciliation-fault mismatch rate in the synthetic QBER sweep.
*Produces:* the **Pearson / Spearman** coefficients.

In [23]:
from scipy.stats import pearsonr, spearmanr

per_qber = synthetic_df.groupby("true_qber").apply(
    lambda g: pd.Series({
        "mismatch_rate": (~g["keys_match"].astype(bool)).mean(),
        "silent_mismatch_rate": ((~g["keys_match"].astype(bool)) & (g.get("verification_passed", True)==True)).mean(),
    })
)
print(per_qber)

r_p, pp = pearsonr(per_qber.index, per_qber["mismatch_rate"])
r_s, ps = spearmanr(per_qber.index, per_qber["mismatch_rate"])
print(f"\nPearson r={r_p:.3f} (p={pp:.3f}), Spearman r={r_s:.3f} (p={ps:.3f})")

           mismatch_rate  silent_mismatch_rate
true_qber                                     
0.0010              0.38                   0.0
0.0026              0.38                   0.0
0.0050              0.54                   0.0
0.0079              0.64                   0.0
0.0105              0.78                   0.0
0.0132              0.94                   0.0
0.0158              0.92                   0.0
0.0200              1.00                   0.0
0.0300              1.00                   0.0

Pearson r=0.893 (p=0.001), Spearman r=0.975 (p=0.000)


In [24]:
check("QBER dependence", "Pearson r", r_p, 0.893, tol=5e-4)
check("QBER dependence", "Spearman r", r_s, 0.975, tol=5e-4)

[PASS] QBER dependence: Pearson r = 0.892725 (expected 0.893, tol 0.0005)
[PASS] QBER dependence: Spearman r = 0.974824 (expected 0.975, tol 0.0005)


### 3d. Cascade leakage model: ξ₁, ξ₂
*Covers:* two **different** calculations that both involve ξ₁, which must not be conflated. (i) A least-squares fit of Cascade leakage to `ξ₁·n·h(Q) + ξ₂·√(n·v(Q))·z`, run on freshly simulated Mock reconciliations; it gives ξ₁ ≈ 2.39 and a **negative, non-physical ξ₂** (the paper remarks on this). (ii) The ξ₁ implied by the idealised finite-key bound for the 5 real keys, ≈ 15.69.
*Produces:* the **ξ₂ negative/non-physical remark** and **ξ₁ ≈ 15.69**.

In [25]:
from qne.cascade.finite_key import h, v
from scipy.stats import norm

class TrackedMockClassicalSession(MockClassicalSession):
    """Drop-in MockClassicalSession that also tallies leaked bits, without
    touching the library source. Remove this and use the library class
    directly once total_leaked_bits is added to qne.cascade permanently."""
    def __init__(self, correct_key):
        super().__init__(correct_key)
        self.total_leaked_bits = 0

    def ask_correct_parities(self, blocks):
        self.total_leaked_bits += len(blocks)
        super().ask_correct_parities(blocks)


def make_synthetic_pair(n_bits, qber, seed):
    alice_synth = Key(nr_bits=n_bits, seed=seed)
    bob_synth = alice_synth.copy()
    bob_synth.apply_noise(bit_error_rate=qber, seed=seed + 1)
    return alice_synth, bob_synth


def measure_epsilon(n_bits, qber, n_trials=200, base_seed=1000):
    failures = 0
    for run in range(n_trials):
        seed = base_seed + run
        a, b = make_synthetic_pair(n_bits, qber, seed)
        session = TrackedMockClassicalSession(correct_key=a)
        reconciliation = Reconciliation(
            algorithm=ORIGINAL, classical_session=session, noisy_key=b,
            estimated_bit_error_rate=qber, seed=seed + 100,
        )
        try:
            bob_reconciled = reconciliation.reconcile()
            if a.nr_bits_different(bob_reconciled) > 0:
                failures += 1
        except RuntimeError:
            failures += 1
    return failures / n_trials


def sweep_leakage(n_values, q_values, n_trials=30, base_seed=2000):
    rows = []
    for n_bits in n_values:
        for qber in q_values:
            leaked = []
            for run in range(n_trials):
                seed = base_seed + run
                a, b = make_synthetic_pair(n_bits, qber, seed)
                session = TrackedMockClassicalSession(correct_key=a)
                reconciliation = Reconciliation(
                    algorithm=ORIGINAL, classical_session=session, noisy_key=b,
                    estimated_bit_error_rate=qber, seed=seed + 100,
                )
                try:
                    bob_reconciled = reconciliation.reconcile()
                    leaked.append(session.total_leaked_bits)
                except RuntimeError:
                    continue
            if leaked:
                rows.append({"n": n_bits, "Q": qber,
                              "mean_leakage": np.mean(leaked), "n_trials_used": len(leaked)})
    return rows


def fit_xi1_xi2(rows, epsilon):
    z = norm.ppf(1 - epsilon)
    X, y = [], []
    for row in rows:
        n_bits, Q, leak = row["n"], row["Q"], row["mean_leakage"]
        X.append([n_bits * h(Q), np.sqrt(n_bits * v(Q)) * z])
        y.append(leak)
    X, y = np.array(X), np.array(y)
    (xi1, xi2), _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    return xi1, xi2, X, y


n_values = [1000, 3000, 5000]
q_values = [0.005, 0.01, 0.02, 0.03, 0.05]

print("Measuring epsilon...")
epsilon = measure_epsilon(n_bits=3400, qber=0.015, n_trials=200)
print(f"epsilon = {epsilon:.4f}")
if epsilon == 0.0:
    print("WARNING: epsilon = 0 exactly -- Phi^-1(1) undefined. "
          "Increase n_trials, or use a conservative published fallback (e.g. 1e-2).")

print("\nSweeping leakage across (n, Q)...")
rows = sweep_leakage(n_values, q_values, n_trials=30)
for r in rows:
    print(f"  n={r['n']}, Q={r['Q']:.3f}: mean_leakage={r['mean_leakage']:.1f} "
          f"({r['n_trials_used']} valid trials)")

print("\nFitting xi1, xi2...")
xi1, xi2, X, y = fit_xi1_xi2(rows, epsilon)
print(f"xi1 = {xi1:.4f}")
print(f"xi2 = {xi2:.4f}")
print(f"Sanity check: published xi1 range is [1.05, 1.16] -- "
      f"{'plausible' if 1.0 <= xi1 <= 1.3 else 'CHECK THIS, seems off'}")

predicted = X @ np.array([xi1, xi2])
residuals = y - predicted
print(f"Mean absolute residual: {np.mean(np.abs(residuals)):.2f} bits")
print(f"Mean relative residual: {np.mean(np.abs(residuals) / y) * 100:.2f}%")

Measuring epsilon...


epsilon = 0.0050

Sweeping leakage across (n, Q)...


  n=1000, Q=0.005: mean_leakage=100.7 (30 valid trials)
  n=1000, Q=0.010: mean_leakage=188.8 (30 valid trials)
  n=1000, Q=0.020: mean_leakage=335.2 (30 valid trials)
  n=1000, Q=0.030: mean_leakage=457.1 (30 valid trials)
  n=1000, Q=0.050: mean_leakage=685.8 (30 valid trials)
  n=3000, Q=0.005: mean_leakage=315.9 (30 valid trials)
  n=3000, Q=0.010: mean_leakage=563.3 (30 valid trials)
  n=3000, Q=0.020: mean_leakage=989.4 (30 valid trials)
  n=3000, Q=0.030: mean_leakage=1372.8 (30 valid trials)
  n=3000, Q=0.050: mean_leakage=2039.2 (30 valid trials)
  n=5000, Q=0.005: mean_leakage=522.8 (30 valid trials)
  n=5000, Q=0.010: mean_leakage=930.1 (30 valid trials)
  n=5000, Q=0.020: mean_leakage=1644.1 (30 valid trials)
  n=5000, Q=0.030: mean_leakage=2289.9 (30 valid trials)
  n=5000, Q=0.050: mean_leakage=3399.9 (30 valid trials)

Fitting xi1, xi2...
xi1 = 2.3948
xi2 = -0.2088
Sanity check: published xi1 range is [1.05, 1.16] -- CHECK THIS, seems off
Mean absolute residual: 6.95 bit

In [26]:
check("Leakage fit (Mock simulation)", "xi1", xi1, 2.3948, tol=5e-4)
check("Leakage fit (Mock simulation)", "xi2 (negative => non-physical)", xi2, -0.2088, tol=5e-4)
assert xi2 < 0

[PASS] Leakage fit (Mock simulation): xi1 = 2.39484 (expected 2.3948, tol 0.0005)
[PASS] Leakage fit (Mock simulation): xi2 (negative => non-physical) = -0.208758 (expected -0.2088, tol 0.0005)


In [27]:
import pandas as pd
import numpy as np


# Fit xi_1 using the idealized formula's residual against the finite-key
# table's known ell values, per Tomamichel & Leverrier's empirical-fit form
def xi_ideal(n, Q):
    h = lambda q: -q*np.log2(q) - (1-q)*np.log2(1-q) if 0 < q < 1 else 0
    v = lambda q: q*(1-q)*(np.log2(q/(1-q)))**2 if 0 < q < 1 else 0
    from scipy.stats import norm
    return 1 + (1/np.sqrt(n)) * np.sqrt(v(Q)/h(Q)) * norm.ppf(1 - 1e-10)

key_meta = toeplitz[["key_index", "qber", "secure_key_length", "k_pe"]].drop_duplicates()
key_meta["n_sifted"] = key_meta["k_pe"] * 10
key_meta["xi_computed"] = key_meta.apply(lambda r: xi_ideal(r["n_sifted"], r["qber"]), axis=1)
print(key_meta)

# xi_1 fit: xi_fit(n) = 1 + xi_1/sqrt(n) -- solve for xi_1 per key, then average
key_meta["xi_1_implied"] = (key_meta["xi_computed"] - 1) * np.sqrt(key_meta["n_sifted"])
print(f"\nMean xi_1: {key_meta['xi_1_implied'].mean():.4f}")

     key_index      qber  secure_key_length  k_pe  n_sifted  xi_computed
0            0  0.012821               2810   390      3900     1.228289
80           1  0.015831               2608   379      3790     1.224333
160          2  0.002674               3180   374      3740     1.280809
240          3  0.002618               3251   382      3820     1.278431
320          4  0.005195               3131   385      3850     1.258011

Mean xi_1: 15.6916


In [28]:
check("Idealised-bound xi1 (different calculation)", "mean xi_1", key_meta["xi_1_implied"].mean(), 15.6916, tol=5e-4)

[PASS] Idealised-bound xi1 (different calculation): mean xi_1 = 15.6916 (expected 15.6916, tol 0.0005)


### 3e. Zero-fault baseline
*Covers:* trials run with no fault injected, to establish the floor the fault results sit on.
*Produces:* the zero-fault baseline count. **2 of 37** asymptotic-mode trials show `keys_match=False` (keys 2 and 4, seed 49, verification failed, 2 residual errors, no faults fired): the baseline is not perfectly clean.

In [29]:
baseline_asym = baseline[baseline["length_mode"] == "asymptotic"]
mismatches = baseline_asym[~baseline_asym["keys_match"].astype(bool)]
print(mismatches[["key_index", "seed", "verification_passed", "remaining_errors_after_reconciliation", "faults_fired"]].to_string())
check("Zero-fault baseline", "trials (asymptotic)", len(baseline_asym), 37)
check("Zero-fault baseline", "mismatches", len(mismatches), 2)

    key_index  seed  verification_passed  remaining_errors_after_reconciliation faults_fired
23          2    49                False                                      2           {}
39          4    49                False                                      2           {}
[PASS] Zero-fault baseline: trials (asymptotic) = 37 (expected 37, tol 1e-06)
[PASS] Zero-fault baseline: mismatches = 2 (expected 2, tol 1e-06)


## 4. Mitigations
### 4a. Redundant privacy amplification
*Covers:* running PA twice with independently seeded Toeplitz faults and comparing the two **outputs** against a clean reference (not just match flags). Real key 0, finite-key output length.
*Produces:* **RPA 100% detection** (every mismatch caught, 0 silent failures).

In [30]:
GEN_ALICE = DATA / "fabric_alice_sifted_bits_genonly.json"
GEN_BOB = DATA / "fabric_bob_sifted_bits_genonly.json"
META_PATH = DATA / "fabric_key0_pe_meta.json"
alice_key, alice_indices = key_from_sifted_json(str(GEN_ALICE), "alice_bits")
bob_key, bob_indices = key_from_sifted_json(str(GEN_BOB), "bob_bits")
assert alice_indices == bob_indices
meta = json.loads(META_PATH.read_text())
k_pe, real_qber = meta["k"], meta["qber"]
n = alice_key.get_nr_bits()
ell_finite, r, t, nu = finite_key_output_length(n, k_pe, real_qber)
print(f"n={n}, k={k_pe}, qber={real_qber:.4f} -> ell_finite={ell_finite}")

n=3488, k=388, qber=0.0077 -> ell_finite=186


In [31]:
FAULT_PROBS = [1e-4, 1e-3, 1e-2, 0.05, 0.1, 0.3, 0.5]  # realistic -> stress-test range
N_RUNS = 40

ext = ToeplitzHashing(input_length=alice_key.get_nr_bits(), output_length=ell_finite)

rows = []
for prob in FAULT_PROBS:
    for run in range(N_RUNS):
        seed = 42 + run
        shared_seed = GF2.Random(ext.seed_length)
        clean_output = np.array(ext.extract(GF2(alice_key.bits), shared_seed))

        injector_1 = SDCFaultInjector(toeplitz_matrix_prob=prob, seed=seed + 100)
        out_1 = injector_1.fast_toeplitz_extract_with_fault(ext, GF2(alice_key.bits), shared_seed, "rep1")

        injector_2 = SDCFaultInjector(toeplitz_matrix_prob=prob, seed=seed + 200)
        out_2 = injector_2.fast_toeplitz_extract_with_fault(ext, GF2(alice_key.bits), shared_seed, "rep2")

        rep1_mismatch = not np.array_equal(out_1, clean_output)
        rep2_mismatch = not np.array_equal(out_2, clean_output)
        any_mismatch = rep1_mismatch or rep2_mismatch

        repeats_agree = np.array_equal(out_1, out_2)
        detected = any_mismatch and not repeats_agree
        silent_failure = any_mismatch and repeats_agree  # both hit the same wrong bit(s) -- vanishingly rare

        rows.append({
            "prob": prob, "run": run,
            "rep1_mismatch": rep1_mismatch, "rep2_mismatch": rep2_mismatch,
            "any_mismatch": any_mismatch,
            "repeats_agree": repeats_agree, "detected": detected,
            "silent_failure": silent_failure,
        })

In [32]:
df_redundant = pd.DataFrame(rows)
print("=== Redundant PA: mismatch rate vs. detection rate ===")
summary_rows = []
for prob in FAULT_PROBS:
    sub = df_redundant[df_redundant["prob"] == prob]
    n_mismatch, n_detected, n_silent = sub["any_mismatch"].sum(), sub["detected"].sum(), sub["silent_failure"].sum()
    summary_rows.append({"prob": prob, "mismatch_rate": n_mismatch / len(sub),
                         "detection_rate": n_detected / n_mismatch if n_mismatch > 0 else float("nan"),
                         "n_silent": int(n_silent)})
df_redundant_summary = pd.DataFrame(summary_rows)
print(df_redundant_summary.to_string(index=False))
check("Redundant PA", "total silent failures", df_redundant["silent_failure"].sum(), 0)
check("Redundant PA", "min detection rate over probs with mismatches", df_redundant_summary["detection_rate"].min(), 1.0)
check("Redundant PA", "total mismatches observed (must be > 0 for the rate to mean anything)", float(df_redundant["any_mismatch"].sum() > 0), 1.0)

=== Redundant PA: mismatch rate vs. detection rate ===
  prob  mismatch_rate  detection_rate  n_silent
0.0001          0.000             NaN         0
0.0010          0.000             NaN         0
0.0100          0.000             NaN         0
0.0500          0.025             1.0         0
0.1000          0.075             1.0         0
0.3000          0.175             1.0         0
0.5000          0.375             1.0         0
[PASS] Redundant PA: total silent failures = 0 (expected 0, tol 1e-06)
[PASS] Redundant PA: min detection rate over probs with mismatches = 1 (expected 1, tol 1e-06)
[PASS] Redundant PA: total mismatches observed (must be > 0 for the rate to mean anything) = 1 (expected 1, tol 1e-06)


### 4b. Triple modular redundancy (TMR)
*Covers:* real-channel TMR on the FABRIC link: 100 voted trials (runs 0-19 from `mitigation_v2`, runs 20-99 from `30_scale_up_experiments`), plus the analytic majority-vote model.
*Produces:* **TMR 0/100 corrected**, the mean residual error count, the **Wilson 95% CI [0, 3.70%]**, and the model prediction that majority voting cannot succeed at p = 0.5 on a pool of 2810 positions but succeeds almost surely at p ≈ 1e-3.
Note that the residual is a **count of error bits** (mean 11.39, SE 0.36), not a percentage.

In [33]:
def wilson_ci(successes, n, z=1.96):
    p_hat = successes / n
    denom = 1 + z**2 / n
    center = (p_hat + z**2 / (2 * n)) / denom
    halfwidth = (z / denom) * np.sqrt(p_hat * (1 - p_hat) / n + z**2 / (4 * n**2))
    return p_hat, max(0, center - halfwidth), min(1, center + halfwidth)

voted = tmr_real[tmr_real["result"] == "voted"]
n_trials, n_corrected = len(voted), int(voted["corrected"].sum())
p_hat, lo, hi = wilson_ci(n_corrected, n_trials)
mean_err = voted["n_errors_after_tmr"].mean()
se_err = voted["n_errors_after_tmr"].std() / np.sqrt(n_trials)
print(f"corrected {n_corrected}/{n_trials}; Wilson 95% CI [{lo*100:.2f}%, {hi*100:.2f}%]; residual error bits {mean_err:.2f} +/- {se_err:.2f} (SE)")
check("TMR (real channel)", "trials", n_trials, 100)
check("TMR (real channel)", "corrected", n_corrected, 0)
check("TMR (real channel)", "Wilson upper bound (%)", hi * 100, 3.70, tol=5e-3)
check("TMR (real channel)", "mean residual error bits", mean_err, 11.39, tol=5e-3)
check("TMR (real channel)", "SE of residual error bits", se_err, 0.36, tol=5e-3)

corrected 0/100; Wilson 95% CI [0.00%, 3.70%]; residual error bits 11.39 +/- 0.36 (SE)
[PASS] TMR (real channel): trials = 100 (expected 100, tol 1e-06)
[PASS] TMR (real channel): corrected = 0 (expected 0, tol 1e-06)
[PASS] TMR (real channel): Wilson upper bound (%) = 3.69948 (expected 3.7, tol 0.005)
[PASS] TMR (real channel): mean residual error bits = 11.39 (expected 11.39, tol 0.005)
[PASS] TMR (real channel): SE of residual error bits = 0.3587 (expected 0.36, tol 0.005)


In [34]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import binom

def prob_position_uncorrected(N, p):
    """Majority vote fails at a given pool position if >= ceil(N/2) of the
    N independent replicas corrupt that same position."""
    threshold = int(np.ceil(N / 2))
    return 1 - binom.cdf(threshold - 1, N, p)  # P(X >= threshold)

def prob_full_correction(N, p, pool_size):
    """All `pool_size` positions must independently survive majority vote."""
    p_fail_per_position = prob_position_uncorrected(N, p)
    return (1 - p_fail_per_position) ** pool_size

def expected_residual_errors(N, p, pool_size):
    return pool_size * prob_position_uncorrected(N, p)

In [35]:
POOL_TOEPLITZ = 2810
check("TMR model", "P(full correction), N=3, p=0.5, pool 2810", prob_full_correction(3, 0.5, POOL_TOEPLITZ), 0.0, tol=1e-12)
check("TMR model", "P(full correction), N=3, p=1e-3, pool 2810", prob_full_correction(3, 1e-3, POOL_TOEPLITZ), 0.99161, tol=5e-5)

[PASS] TMR model: P(full correction), N=3, p=0.5, pool 2810 = 0 (expected 0, tol 1e-12)
[PASS] TMR model: P(full correction), N=3, p=1e-3, pool 2810 = 0.991611 (expected 0.99161, tol 5e-05)


### 4c. Mitigation overhead (timing)
*Covers:* wall-clock cost of redundant PA (2 extractions vs 1) and TMR (3 reconciliations vs 1) on this machine.
*Produces:* the **1.08x** and **2.69x** overhead ratios. These are timings, so they are re-measured and compared with a tolerance; they are the only non-exact checks.
**Finding:** the 1.08x does **not** reproduce. Two extractions cannot cost 1.08x of one; in steady state the ratio is ~2.0x. The original single-extraction mean (1.32 ms) is dominated by one-time first-call warm-up averaged into 30 trials, which hides most of the redundant PA cost. The cell below reports both the original-method ratio (a MISMATCH against 1.08x) and the steady-state ratio. The paper's 1.08x should be re-examined.

In [36]:
N_TRIALS = 30
krow = key_pairs_live.iloc[0]
n_bits, ell = int(krow["n_bits"]), int(krow["n_bits"] * 0.9)  # adjust ell to your real output length

alice_bits = GF2(np.random.default_rng(0).integers(0, 2, n_bits))
ext = ToeplitzHashing(input_length=n_bits, output_length=ell)

single_times, redundant_times = [], []
for trial in range(N_TRIALS):
    seed_gf2 = GF2.Random(ext.seed_length)

    t0 = time.perf_counter()
    _ = ext.extract(alice_bits, seed_gf2)
    single_times.append(time.perf_counter() - t0)

    t0 = time.perf_counter()
    _ = ext.extract(alice_bits, seed_gf2)
    _ = ext.extract(alice_bits, seed_gf2)   # the second, redundant extraction
    redundant_times.append(time.perf_counter() - t0)

single_mean, redundant_mean = np.mean(single_times), np.mean(redundant_times)
print(f"Single PA extraction:    {single_mean*1000:.2f} ms  (n={N_TRIALS})")
print(f"Redundant (2x) PA:       {redundant_mean*1000:.2f} ms  (n={N_TRIALS})")
print(f"Overhead ratio:          {redundant_mean/single_mean:.2f}x")

Single PA extraction:    0.09 ms  (n=30)
Redundant (2x) PA:       0.17 ms  (n=30)
Overhead ratio:          1.89x


In [37]:
rpa_ratio = redundant_mean / single_mean
check("Overhead", "redundant PA ratio, original method (paper: 1.08x)", rpa_ratio, 1.08, tol=0.25, kind="timing")
# Steady state: drop the first 5 trials (one-time warm-up / caching) and compare again.
rpa_steady = np.mean(redundant_times[5:]) / np.mean(single_times[5:])
print(f"first single extraction: {single_times[0]*1000:.2f} ms vs. steady-state median {np.median(single_times[5:])*1000:.2f} ms")
check("Overhead", "redundant PA ratio, steady state (two extractions => ~2x)", rpa_steady, 2.0, tol=0.25, kind="timing")

[MISMATCH] Overhead: redundant PA ratio, original method (paper: 1.08x) = 1.89001 (expected 1.08, tol 0.25)
first single extraction: 0.22 ms vs. steady-state median 0.08 ms
[PASS] Overhead: redundant PA ratio, steady state (two extractions => ~2x) = 1.98598 (expected 2, tol 0.25)


In [38]:
GEN_ALICE = DATA / "fabric_alice_sifted_bits_genonly.json"
GEN_BOB = DATA / "fabric_bob_sifted_bits_genonly.json"
if not (GEN_ALICE.exists() and GEN_BOB.exists()):
    raise FileNotFoundError("Adjust these paths to wherever your real key pair's sifted bits live.")

alice_key_ov, _ = key_from_sifted_json(str(GEN_ALICE), "alice_bits")
bob_key_ov, _ = key_from_sifted_json(str(GEN_BOB), "bob_bits")
qber = 0.0128  # use this key's real measured QBER

N_TRIALS = 10  # reconciliation is slower than PA extraction -- fewer trials is fine
single_times, tmr_times = [], []

for trial in range(N_TRIALS):
    seed = trial

    t0 = time.perf_counter()
    session = MockClassicalSession(correct_key=alice_key_ov)
    recon = Reconciliation(algorithm=ORIGINAL, classical_session=session, noisy_key=bob_key_ov,
                             estimated_bit_error_rate=qber, seed=seed, correct_key=alice_key_ov)
    recon.reconcile()
    single_times.append(time.perf_counter() - t0)

    t0 = time.perf_counter()
    for rep in range(3):
        session = MockClassicalSession(correct_key=alice_key_ov)
        recon = Reconciliation(algorithm=ORIGINAL, classical_session=session, noisy_key=bob_key_ov,
                                 estimated_bit_error_rate=qber, seed=seed + rep * 1000, correct_key=alice_key_ov)
        recon.reconcile()
    tmr_times.append(time.perf_counter() - t0)

single_mean, tmr_mean = np.mean(single_times), np.mean(tmr_times)
print(f"Single reconciliation:   {single_mean*1000:.1f} ms  (n={N_TRIALS})")
print(f"TMR (3x reconciliation): {tmr_mean*1000:.1f} ms  (n={N_TRIALS})")
print(f"Overhead ratio:          {tmr_mean/single_mean:.2f}x")

Single reconciliation:   5.5 ms  (n=10)
TMR (3x reconciliation): 16.7 ms  (n=10)
Overhead ratio:          3.03x


In [39]:
tmr_ratio = tmr_mean / single_mean
check("Overhead", "TMR ratio (expected 2.69x, timing-dependent; structurally ~3x)", tmr_ratio, 2.69, tol=0.5, kind="timing")

[PASS] Overhead: TMR ratio (expected 2.69x, timing-dependent; structurally ~3x) = 3.02772 (expected 2.69, tol 0.5)


## 5. Finite-key length
*Covers:* secure output length `ℓ_finite` from the finite-key bound vs. the naive asymptotic `n(1-2h(Q))`, per key.
*Produces:* **Table 5**. Computed from the **5-key snapshot** `key_pairs_metadata_table5_5keys.csv`; the live 10-key file is shown only to document why it must not be used for this table.

In [40]:
def asymptotic_key_length(n_bits, Q):
    return max(0, int(n_bits * (1 - 2 * h(Q))))

def table5(meta):
    rows = []
    for _, row in meta.iterrows():
        n_bits, k_pe_, Q = int(row["n_bits"]), int(row["k_pe"]), float(row["qber"])
        ell_a = asymptotic_key_length(n_bits, Q)
        ell_f, _, _, _ = finite_key_output_length(n_bits, k_pe_, Q)
        rows.append({"key": int(row["index"]), "qber": Q, "ell_asymptotic": ell_a, "ell_finite": ell_f,
                     "reduction_%": 100 * (1 - ell_f / ell_a) if ell_a > 0 else float("nan")})
    return pd.DataFrame(rows)

t5 = table5(key_pairs_table5)
print(t5.round(2).to_string(index=False))
print(f"\nmean reduction: {t5['reduction_%'].mean():.2f}%")
expected_rows = {0: 99.18, 1: 99.96, 2: 89.40, 3: 88.83, 4: 91.41}
for k, v in expected_rows.items():
    check("Table 5", f"reduction % key {k}", t5.loc[t5["key"] == k, "reduction_%"].iloc[0], v, tol=5e-3)
check("Table 5", "mean reduction % (5 keys)", t5["reduction_%"].mean(), 93.76, tol=5e-3)
check("Table 5", "rows", len(t5), 5)

t5_live = table5(key_pairs_live)
print(f"\n(for contrast) live 10-key file: mean reduction {t5_live['reduction_%'].mean():.2f}% over {len(t5_live)} rows -- NOT the paper's Table 5")
check("Table 5 (contrast)", "live 10-key mean reduction %", t5_live["reduction_%"].mean(), 96.05, tol=5e-3, kind="contrast")

 key  qber  ell_asymptotic  ell_finite  reduction_%
   0  0.01            2810          23        99.18
   1  0.02            2608           1        99.96
   2  0.00            3180         337        89.40
   3  0.00            3251         363        88.83
   4  0.01            3131         269        91.41

mean reduction: 93.76%
[PASS] Table 5: reduction % key 0 = 99.1815 (expected 99.18, tol 0.005)
[PASS] Table 5: reduction % key 1 = 99.9617 (expected 99.96, tol 0.005)
[PASS] Table 5: reduction % key 2 = 89.4025 (expected 89.4, tol 0.005)
[PASS] Table 5: reduction % key 3 = 88.8342 (expected 88.83, tol 0.005)
[PASS] Table 5: reduction % key 4 = 91.4085 (expected 91.41, tol 0.005)
[PASS] Table 5: mean reduction % (5 keys) = 93.7577 (expected 93.76, tol 0.005)
[PASS] Table 5: rows = 5 (expected 5, tol 1e-06)

(for contrast) live 10-key file: mean reduction 96.05% over 10 rows -- NOT the paper's Table 5
[PASS] Table 5 (contrast): live 10-key mean reduction % = 96.05 (expected 96.05,

## 6. Figures
*Covers:* regeneration of each paper figure from the committed data, each under an **explicit** matplotlib style (`rc_context`), written to a temporary directory and compared pixel-by-pixel with `results/figures/`.
*Produces:* **Fig 8** (model fit), **Fig 6a** (naive vs. proposed), **Fig 7** (redundancy), **Fig 5** (mock vs. real), **Fig 9** (synthetic QBER sweep), **Fig 4** (mismatch rate and QBER, all fault types), **Fig 6b** (pooled reconciliation fit).
Figure numbering is inherited from `PAPER_TRACEBACK.md`; the paper text itself is not in the repository.
Fig 7's bar heights are literal constants in the original cell (`[1.0, 0.0]`, `[1.08, 2.69]`), not computed from data; they are reproduced as written.

In [41]:
# names the figure cells below expect (originally defined in paper_figures_and_reconciliation_fit c0)
def silent_mismatch_rate(g):
    return ((~g["keys_match"].astype(bool)) & (g["verification_passed"] == True)).mean()

In [42]:
# Fig 8
with plt.rc_context(STYLE_C0):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    predictors = [
        (toeplitz, lambda p, ell, t: p * (ell / (ell + t)) * 0.5),
        (final_key, lambda p, ell, t: p * (ell / (ell + t))),
    ]
    for ax, (df, predictor) in zip(axes, predictors):
        probs = sorted(df["prob"].unique())
        obs, pred = [], []
        for p in probs:
            sub = df[df["prob"] == p]
            obs.append(silent_mismatch_rate(sub))
            pred.append(sub.apply(lambda r: predictor(p, r["secure_key_length"], r["t_verify"]), axis=1).mean())
        ax.plot(probs, obs, "o-", label="observed")
        ax.plot(probs, pred, "s--", label="predicted")
        ax.set_xscale("log")
        ax.set_xlabel("firing probability")
        ax.set_ylabel("silent mismatch rate")
        ax.legend()
    plt.tight_layout()
    plt.savefig(str(REPRO / "fig_model_fit.png"), dpi=150, bbox_inches="tight")
    plt.close()
plt.close('all')
compare_png('Fig 8', 'fig_model_fit.png')

[PASS] Fig 8: fig_model_fit.png differing pixels vs committed = 0


In [43]:
# Fig 6a
with plt.rc_context(STYLE_C0):
    fig, ax = plt.subplots(figsize=(6, 4))
    probs = sorted(toeplitz["prob"].unique())
    obs = [silent_mismatch_rate(toeplitz[toeplitz["prob"] == p]) for p in probs]
    naive = probs
    proposed = [toeplitz[toeplitz["prob"] == p].apply(
        lambda r: p * (r["secure_key_length"] / (r["secure_key_length"] + r["t_verify"])) * 0.5, axis=1
    ).mean() for p in probs]
    ax.plot(probs, obs, "o-", label="observed")
    ax.plot(probs, naive, ":", label="naive (P=p_f)")
    ax.plot(probs, proposed, "s--", label="proposed model")
    ax.set_xscale("log")
    ax.set_xlabel("firing probability")
    ax.set_ylabel("mismatch rate")
    ax.legend()
    plt.tight_layout()
    plt.savefig(str(REPRO / "fig_naive_vs_proposed.png"), dpi=150, bbox_inches="tight")
    plt.close()
plt.close('all')
compare_png('Fig 6a', 'fig_naive_vs_proposed.png')

[PASS] Fig 6a: fig_naive_vs_proposed.png differing pixels vs committed = 0


In [44]:
# Fig 7
with plt.rc_context(STYLE_C0):
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].bar(["Redundant PA", "TMR"], [1.0, 0.0])
    axes[0].set_ylabel("detection/correction rate")
    axes[1].bar(["Redundant PA", "TMR"], [1.08, 2.69])
    axes[1].set_ylabel("overhead (x)")
    plt.tight_layout()
    plt.savefig(str(REPRO / "fig_redundancy.png"), dpi=150, bbox_inches="tight")
    plt.close()
plt.close('all')
compare_png('Fig 7', 'fig_redundancy.png')

[PASS] Fig 7: fig_redundancy.png differing pixels vs committed = 0


In [45]:
# Fig 5
with plt.rc_context(STYLE_C0):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, (df, fault_type) in zip(axes, [(toeplitz, "toeplitz"), (final_key, "final_key")]):
        probs = sorted(df["prob"].unique())
        real_rates = [silent_mismatch_rate(df[df["prob"] == p]) for p in probs]
        mock_sub = mock_asym[mock_asym["fault_type"] == fault_type]
        mock_rates = [silent_mismatch_rate(mock_sub[mock_sub["prob"] == p]) for p in probs]
        ax.plot(probs, real_rates, "o-", label="real channel")
        ax.plot(probs, mock_rates, "s--", label="mock")
        ax.set_xscale("log")
        ax.set_xlabel("firing probability")
        ax.set_ylabel("silent mismatch rate")
        ax.legend()
    plt.tight_layout()
    plt.savefig(str(REPRO / "fig_mock_vs_real.png"), dpi=150, bbox_inches="tight")
    plt.close()
plt.close('all')
compare_png('Fig 5', 'fig_mock_vs_real.png')

[PASS] Fig 5: fig_mock_vs_real.png differing pixels vs committed = 0


In [46]:
# Fig 9
with plt.rc_context(STYLE_C0):
    per_qber = synthetic_df.groupby("true_qber").apply(lambda g: (~g["keys_match"].astype(bool)).mean())
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(per_qber.index, per_qber.values, "o-", color="darkorange")
    ax.set_xlabel("QBER")
    ax.set_ylabel("mismatch rate")
    plt.tight_layout()
    plt.savefig(str(REPRO / "fig_synthetic_qber_sweep.png"), dpi=150, bbox_inches="tight")
    plt.close()
plt.close('all')
compare_png('Fig 9', 'fig_synthetic_qber_sweep.png')

[PASS] Fig 9: fig_synthetic_qber_sweep.png differing pixels vs committed = 0


In [47]:
# Fig 4 (sets its own, larger style)
with plt.rc_context():
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    from pathlib import Path

    PROJECT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
    RESULTS = PROJECT_DIR / "results"

    plt.rcParams.update({
        "font.size": 20,
        "axes.labelsize": 22,
        "xtick.labelsize": 18,
        "ytick.labelsize": 18,
        "legend.fontsize": 16,
        "lines.linewidth": 3,
        "lines.markersize": 10,
    })

    def silent_mismatch_rate(g):
        return ((~g["keys_match"].astype(bool)) & (g["verification_passed"] == True)).mean()

    toeplitz = pd.read_csv(RESULTS / "data" / "sdc_realchannel_toeplitz_allkeys_asymptotic.csv")
    final_key = pd.read_csv(RESULTS / "data" / "sdc_realchannel_finalkey_allkeys_asymptotic.csv")
    reconciliation = pd.read_csv(RESULTS / "data" / "reconciliation_clean.csv")

    MISMATCH_COLOR = "tab:blue"
    QBER_COLOR = "tab:red"
    MISMATCH_YLIM = (0, 0.5)  # consistent across all three rows

    fig, axes = plt.subplots(3, 1, figsize=(9, 17))
    row_labels = ["a) Toeplitz-matrix fault", "b) Final-key fault", "c) Reconciliation-state fault"]
    fault_data = [toeplitz, final_key, reconciliation]

    for ax_left, df, label in zip(axes, fault_data, row_labels):
        probs = sorted(df["prob"].unique())

        mismatch_by_prob = [silent_mismatch_rate(df[df["prob"] == p]) for p in probs]
        qber_by_prob = [df[df["prob"] == p]["qber"].mean() for p in probs]

        ax_left.plot(probs, mismatch_by_prob, "o-", color=MISMATCH_COLOR, label="silent mismatch rate")
        ax_left.set_xscale("log")
        ax_left.set_xlabel("firing probability")
        ax_left.set_ylabel("silent mismatch rate", color=MISMATCH_COLOR)
        ax_left.tick_params(axis="y", labelcolor=MISMATCH_COLOR)
        ax_left.set_ylim(*MISMATCH_YLIM)

        ax_right = ax_left.twinx()
        ax_right.plot(probs, qber_by_prob, "s--", color=QBER_COLOR, label="QBER")
        ax_right.set_ylabel("QBER", color=QBER_COLOR)
        ax_right.tick_params(axis="y", labelcolor=QBER_COLOR)
        ax_right.set_ylim(0, 0.11)

        # Anchored to this axes' OWN coordinate system (0-1 fraction of its own
        # width/height), not a figure-level position -- this stays correct
        # regardless of what tight_layout/subplots_adjust does afterward, since
        # it moves WITH the axes rather than being computed once beforehand.
        ax_left.text(0.5, -0.38, label, transform=ax_left.transAxes,
                     ha="center", va="top", fontsize=22, fontweight="bold")

    # Generous vertical gap so each row's label has clear space below its
    # x-axis and above the next row's title -- set BEFORE relying on any
    # position, not after.
    plt.subplots_adjust(hspace=0.7)
    plt.savefig(str(REPRO / "fig_mismatch_prob_and_qber_all_faults.png"), dpi=150, bbox_inches="tight")
    plt.close()
plt.close('all')
compare_png('Fig 4', 'fig_mismatch_prob_and_qber_all_faults.png')

[PASS] Fig 4: fig_mismatch_prob_and_qber_all_faults.png differing pixels vs committed = 0


In [48]:
# Fig 6b: pooled reconciliation fit (needs df_baseline_fit from section 3b)
with plt.rc_context(STYLE_C0):
    def plot_reconciliation_fit_pooled(reconciliation_clean, key_pairs_df, df_baseline_fit, save=True):
        per_key = []
        for key_idx in sorted(reconciliation_clean["key_index"].unique()):
            sub = reconciliation_clean[reconciliation_clean["key_index"] == key_idx]
            c_fixed = sub["total_corrections"].mean()
            b_fit = df_baseline_fit.loc[df_baseline_fit["key_index"] == key_idx, "b_fit (baseline error rate)"].values[0]
            per_key.append({"key_index": key_idx, "c": c_fixed, "b": b_fit})
        per_key_df = pd.DataFrame(per_key)

        mean_c = per_key_df["c"].mean()
        mean_b = per_key_df["b"].mean()

        pooled_obs = reconciliation_clean.groupby("reconciliation_prob")["mismatch"].mean()
        p_vals, obs = pooled_obs.index.values, pooled_obs.values

        p_smooth = np.logspace(np.log10(p_vals.min()), np.log10(p_vals.max()), 200)
        naive_curve = p_smooth  # P(D) = p_f -- one firing assumed to directly cause one mismatch
        proposed_curve = 1 - (1 - mean_b) * (1 - p_smooth) ** mean_c

        plt.rcParams.update({"font.size": 13})
        fig, ax = plt.subplots(figsize=(6, 4.5))
        ax.plot(p_vals, obs, "o-", color="#2E5FA3", label="observed (pooled, all keys)", markersize=7, linewidth=2)
        ax.plot(p_smooth, naive_curve, ":", color="#E67E22", label=r"naive ($P=p_f$)", linewidth=2.2)
        ax.plot(p_smooth, proposed_curve, "--", color="#3F8F5F", label="proposed model", linewidth=2.2)

        ax.set_xscale("log")
        ax.set_xlabel("firing probability")
        ax.set_ylabel("mismatch rate")
        ax.set_ylim(-0.03, 1.03)
        ax.legend(loc="upper left", fontsize=10.5)
        fig.tight_layout()
        if save:
            fig.savefig(str(REPRO / "fig_reconciliation_fit_pooled.png"), dpi=200)
        plt.close(fig)

        print(f"mean c (measured) = {mean_c:.2f}, mean b = {mean_b:.4f}")
        return per_key_df

    per_key_df = plot_reconciliation_fit_pooled(reconciliation_clean, key_pairs_df, df_baseline_fit)
    print(per_key_df)
plt.close('all')
compare_png('Fig 6b', 'fig_reconciliation_fit_pooled.png')

mean c (measured) = 33.11, mean b = 0.0783
   key_index         c             b
0          0  32.79625  1.992909e-02
1          1  34.86875  3.906122e-07
2          2  32.50125  1.925303e-01
3          3  31.96000  1.353539e-01
4          4  33.42375  4.359409e-02
[PASS] Fig 6b: fig_reconciliation_fit_pooled.png differing pixels vs committed = 0


## 7. Summary
*Covers:* every check performed above in one table, and a final assertion.
*Produces:* the reproduction verdict. Exact numeric checks and figure comparisons must all pass; timing checks are reported separately because they depend on the machine.

In [49]:
summary = pd.DataFrame(CHECKS)
pd.set_option("display.width", 200, "display.max_colwidth", 70, "display.max_rows", 500)
print(summary.groupby(["kind", "status"]).size().unstack(fill_value=0))
print()
print(summary[summary["status"] != "PASS"].to_string(index=False) if (summary["status"] != "PASS").any() else "No mismatches.")
required = summary[summary["kind"].isin(["exact", "figure"])]
assert (required["status"] == "PASS").all(), "some cited results did NOT reproduce, see table above"
print(f"\nAll {len(required)} exact and figure checks reproduced; {int((summary['kind'] == 'timing').sum())} timing checks reported separately.")
summary

status    MISMATCH  PASS
kind                    
contrast         0     1
exact            0    64
figure           0     7
timing           1     2

artifact                                           quantity  reproduced  expected  tol   kind   status
Overhead redundant PA ratio, original method (paper: 1.08x)    1.890008      1.08 0.25 timing MISMATCH

All 71 exact and figure checks reproduced; 3 timing checks reported separately.


,artifact,quantity,reproduced,expected,tol,kind,status
0,Data,toeplitz rows,1.200000e+03,1.200000e+03,1.000000e-06,exact,PASS
1,Data,final_key rows,1.200000e+03,1.200000e+03,1.000000e-06,exact,PASS
2,Data,reconciliation rows,4.000000e+03,4.000000e+03,1.000000e-06,exact,PASS
3,Data,TMR real-channel runs,1.000000e+02,1.000000e+02,1.000000e-06,exact,PASS
4,Table 4,toeplitz silent rate p=0.1,3.333333e-02,3.300000e-02,5.000000e-04,exact,PASS
5,Table 4,final_key silent rate p=0.1,6.666667e-02,6.700000e-02,5.000000e-04,exact,PASS
6,Table 4,reconciliation silent rate p=0.1,0.000000e+00,0.000000e+00,5.000000e-04,exact,PASS
7,Table 4,toeplitz silent rate p=0.3,1.266667e-01,1.270000e-01,5.000000e-04,exact,PASS
8,Table 4,final_key silent rate p=0.3,2.333333e-01,2.330000e-01,5.000000e-04,exact,PASS
9,Table 4,reconciliation silent rate p=0.3,0.000000e+00,0.000000e+00,5.000000e-04,exact,PASS
